# Download Sentinel-2 images from AWS and crop

Requires AWS and GDAL command line functionality.

In [ ]:
# !pip install mgrs # Military Grid Reference System for Sentinel 2 tiling
# !pip install utm #functions to convert to/from UTM

In [1]:
import os
import subprocess
import numpy as np
import shutil
import geopandas as gpd
import fiona
import shapely
import glob
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import mgrs
import utm
from pyproj import CRS

# # Enable fiona KML file reading driver
# fiona.drvsupport.supported_drivers['LIBKML'] = 'rw'

In [10]:
#specify site name & root path
projectpath = '/Users/ellynenderlin/Research/NSF_GrIS-Freshwater/melange/'
site = 'UMS'
basepath = projectpath+site+'/'
print('Site path: ',basepath)

# set impath where the images will be downloaded
# impath = '/Volumes/JUKES_EXT/CautoRIFT_sites/S2_B3/'
# impath = '/Users/ellynenderlin/Research/NASA_CryoIdaho/glaciers/Wolverine/images/S2/'
# impath = '/Volumes/Jokulhaup_5T/Greenland-melange/ZIM/images/S2/'
impath = basepath+'images/S2/'
print('Image path: ',impath)

# impath = '/Users/jukesliu/Documents/TURNER/DATA/IMAGERY/sentinel2/'
# impath = '/Users/jukesliu/Documents/GOI_Alaska/fieldmap/images/'

Site path:  /Users/ellynenderlin/Research/NSF_GrIS-Freshwater/melange/UMS/
Image path:  /Users/ellynenderlin/Research/NSF_GrIS-Freshwater/melange/UMS/images/S2/


# 0) Automatically the identify S2 tile that overlaps your AOI

Must download the S2 tiling grid from here: https://sentiwiki.copernicus.eu/web/s2-products

In [11]:
# AOIpath = '/Volumes/JUKES_EXT/AUTO-TERMINUS-TRACES/Antarctic-test/BoxThwaites/BufferThwaites_PS3031.shp' # path to the AOI rectangle shapefile
# AOIpath = '/Users/jukesliu/Documents/PECLET/B3/B3/B3_Box_WGS.shp' 
# AOIpath = '/Users/ellynenderlin/Research/NASA_CryoIdaho/glaciers/Wolverine/AOIs/Wolverine-2018-box-WGS.shp' 
# AOIpath = '/Volumes/Jokulhaup_5T/Greenland-melange/ZIM/shapefiles/ZIM-melange-box_WGS.shp' 
AOIpath = basepath+'shapefiles/'+site+'-melange-box_WGS.shp'

# # set the path to the S2 tiling grid KML file: NOT NEEDED IF YOU USE THE MGRS OPTION!
# S2grid_path = '/Users/ellynenderlin/Research/miscellaneous/S2A_OPER_GIP_TILPAR_MPC__20151209T095117_V20150622T000000_21000101T000000_B00.kml'

In [12]:
# SET YEARS AND MONTHS TO DOWNLOAD
years = np.arange(2019,2026); print(years) # set year(s) to download
months = np.linspace(1,12,12); print(months) # ANNUAL: set month(s) to download
# months = np.linspace(4,10,7); print(months) # SUMMER: set month(s) to download
outputpath = impath # outputpath will be in the impath folder

[2019 2020 2021 2022 2023 2024 2025]
[ 1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12.]


In [30]:
## Reproject AOI box to WGS84 and/or UTM as needed  ####################################
AOI = fiona.open(AOIpath).next()
aoi = AOI['geometry']['coordinates'][0]
# print(aoi)

# automatically extract the source coordinate reference system from the input file
from fiona.crs import to_string
with fiona.open(AOIpath) as colxn:
    source_srs_str = to_string(colxn.crs)
source_srs = source_srs_str[6:]
print(source_srs)
# manually enter source_srs if needed
# source_srs = '32606' # EPSG code for the current projection of the glacier shapefile(s)

# Reproject to WGS if in a different srs (needed to find the tile index)
if not source_srs.endswith('4326'):
    print('Reprojecting AOI file')
    #IF GENERIC NAME: "BoxID" with number
    # boxespath = impath+"Box"+BoxID+"/Box"+BoxID # access the BoxID folders created 
    # rp = "ogr2ogr -f 'ESRI Shapefile' -t_srs EPSG:4326 -s_srs EPSG:"+source_srs+" "
    # rp +=boxespath+"_WGS.shp "+boxespath+".shp"
    #IF CUSTOM NAME (NOTE: customized indexing based on input AOI name)
    if source_srs.startswith('epsg'):
        rp = "ogr2ogr -f 'ESRI Shapefile' -t_srs EPSG:4326 -s_srs "+source_srs+" "
    else:
        rp = "ogr2ogr -f 'ESRI Shapefile' -t_srs EPSG:4326 -s_srs EPSG:"+source_srs+" "
    rp +=AOIpath[:-7]+"WGS.shp "+AOIpath 
    
    # check the command and run it
    print("Command:", rp) # check command
    subprocess.run(rp, shell=True, check=True) # run the command on terminal
    AOIpath = AOIpath[:-7]+"WGS.shp"
else:
    print('AOI already in EPSG:4326')

# read in the WGS84 AOI file
aoi_gdf = gpd.read_file(AOIpath) # read in the WGS-version of the AOI shapefile
aoi = aoi_gdf.geometry.values
# print(aoi)

# Reproject AOI to UTM (needed to crop images)
centroid = aoi[0].centroid
longitude = centroid.x; latitude = centroid.y # Extract the latitude (y-coordinate) and longitude (x-coordinate)
easting, northing, zone_number, zone_letter = utm.from_latlon(latitude, longitude)
UTMzone = zone_number
print('UTM zone: ',zone_number)
if latitude >= 0:
    crs = CRS.from_dict({'proj': 'utm', 'zone': int(zone_number), 'north': True})
else:
    crs = CRS.from_dict({'proj': 'utm', 'zone': int(zone_number), 'south': True})
print(crs.to_authority())
if 'WGS' in AOIpath:
    rp = "ogr2ogr -f 'ESRI Shapefile' -t_srs EPSG:"+crs.to_authority()[1]+" -s_srs EPSG:4326 "
    rp +=AOIpath[:-7]+"UTM"+str(zone_number)+".shp "+AOIpath 
    boxpath = AOIpath[:-7]+"UTM"+str(zone_number)+".shp"
else:
    rp = "ogr2ogr -f 'ESRI Shapefile' -t_srs EPSG:"+crs.to_authority()[1]+" -s_srs EPSG:4326 "
    rp +=AOIpath[:-4]+"-UTM"+str(zone_number)+".shp "+AOIpath 
    boxpath = AOIpath[:-4]+"UTM"+str(zone_number)+".shp"
subprocess.run(rp, shell=True, check=True) # run the command on terminal

# # if an error is produced, check the error output on the terminal window that runs this notebook
######################################################################################

epsg:4326
AOI already in EPSG:4326
UTM zone:  22
('EPSG', '32622')


/var/folders/n8/48j91knn5_5b_3r1m75mfwp0n0j254/T/ipykernel_50792/3876022692.py:2: FionaDeprecationWarning: Collection.__next__() is buggy and will be removed in Fiona 2.0. Switch to `next(iter(collection))`.
  AOI = fiona.open(AOIpath).next()
Warning 1: Value 225706935.0957793 of field AREA of feature 0 not successfully written. Possibly due to too larger number with respect to field width


CompletedProcess(args="ogr2ogr -f 'ESRI Shapefile' -t_srs EPSG:32622 -s_srs EPSG:4326 /Users/ellynenderlin/Research/NSF_GrIS-Freshwater/melange/UMS/shapefiles/UMS-melange-box_UTM22.shp /Users/ellynenderlin/Research/NSF_GrIS-Freshwater/melange/UMS/shapefiles/UMS-melange-box_WGS.shp", returncode=0)

## Choose one of three options for identifying the correct tile:

In [31]:
# OPTION 1: FIND THE MGRS TILE THAT OVERLAPS THE CENTROID OF THE POLYGON
# Get the centroid
centroid = aoi[0].centroid

# Extract the latitude (y-coordinate) and longitude (x-coordinate)
longitude = centroid.x
latitude = centroid.y
print(f"Latitude: {latitude}, Longitude: {longitude}")

# Create an MGRS converter object
m = mgrs.MGRS()

# Convert to MGRS
mgrs_coordinate = m.toMGRS(latitude,longitude)

# Print the result
print(f"MGRS Coordinate: {mgrs_coordinate}")
tilename = [mgrs_coordinate[0:5]]
print(tilename)
tilefolder = mgrs_coordinate[0:2]+'/'+mgrs_coordinate[2:3]+'/'+mgrs_coordinate[3:5]+'/'
print(tilefolder)

Latitude: 71.69990191257094, Longitude: -52.53761044405324
MGRS Coordinate: 22WDE4611856146
['22WDE']
22/W/DE/


In [ ]:
# OPTION 2: USE THE SENTINEL 2 TILING KML THAT YOU DOWNLOADED TO FIND ALL OVERLAPPING TILES (OBSOLETE! DOESN'T ALWAYS WORK)
# Load the S2 tile grid and plot it
s2grid_shp = fiona.open(S2grid_path) # open the S2 tile grid

# find the S2 footprint tile(s) overlapping the AOI
# tilename = 'None'
tilename = []
for feature in s2grid_shp:
    tile = shapely.geometry.Polygon(feature['geometry']['geometries'][0]['coordinates'][0])
#     print(feature['properties']['Name'])
    if tile.overlaps(aoi[0]):
        # tilename=feature['properties']['Name']
        tempname=feature['properties']['Name']
        tilename.append(tempname)
        # print(tilename)
        
        # break # stop searching
print(tilename)

# check that the folder paths will be correct (Ex: 21/X/VC or 21/X/WD)
# tilefolder = str(int(tilename[0:2]))+'/'+tilename[2:3]+'/'+tilename[3:]+'/'
for tile in tilename:
    tilefolder = tile[0:2]+'/'+tile[2:3]+'/'+tile[3:]+'/'
    print(tilefolder)

## Or set it manually here:

In [ ]:
# OPTION 3: MANUALLY SPECIFY THE TILE(S)

# specify the tile name or names as ['##XLL','##XLL']
tilename = ['21XWC'] #Alison Glacier.... need to figure out why the automated tile overlap search didn't work
# tilename = ['27XWH'] #Zachariae Isstrom
# tilename = ['22WEB'] #Sermeq Kujalleq

for tile in tilename:
    tilefolder = tile[0:2]+'/'+tile[2:3]+'/'+tile[3:]+'/'
    print(tilefolder)

#deprecated: specify the tile folder name
# tilefolder = '13/C/DS/'
# tilefolder = '7/V/EG/'

## Explore files available on AWS manually:

In [ ]:
# # explore files manually:
# !export PATH=/usr/local/bin:/usr/bin:/bin:/usr/sbin:$PATH; aws s3 ls s3://sentinel-cogs/sentinel-s2-l2a-cogs/35/X/MH/2024/ --no-sign-request

In [ ]:
# # try filtering by clouds
# from pystac_client import Client
# client = Client.open(api_url)
# collection = "sentinel-2-l2a"

# search = client.search(
#     collections=[collection],
#     MGRS_TILE='6VUM'
#     datetime="2020-03-20/2020-03-30",
#     query=["eo:cloud_cover<50"]
# )
# print(search.matched())

# 1) Download data masks

Syntax:

aws --no-sign-request s3 cp s3://landsat-pds/c1/L8/031/005/ Output/path/LS8aws/Path031_Row005/ --recursive --exclude "*" --include "*MTL.txt"

In [ ]:
# #OBSOLETE: Variables are defined during AOI reprojection

# # list the UTM zone as number (used for indexing of the download path)
# UTMzone = 24

# # Specify the AOI extent (aka box) in UTM coordinates (used for cropping)
# # boxpath = AOIpath #  path to the AOI Box (UTM!!)
# # imagepath = '/Users/jukesliu/Documents/GOI_Alaska/fieldmap/images/' # path to the downloaded images
# # boxpath = '/Users/jukesliu/Documents/TURNER/DATA/shapefiles_gis/BoxTurner_UTM_07.shp'  #  path to the AOI Box (UTM!!)
# # boxpath = '/Users/ellynenderlin/Research/NASA_CryoIdaho/glaciers/Wolverine/AOIs/Wolverine-2018-box-WGS_UTM_06.shp'   #  path to the AOI Box (UTM!!)
# # boxpath = '/Volumes/Jokulhaup_5T/Greenland-melange/ZIM/shapefiles/ZIM-melange-box_UTM27.shp' 
# # boxpath = '/Volumes/Jokulhaup_5T/Greenland-melange/ZIM/shapefiles/ZIM-melange-box_UTM27.shp' 
# boxpath = basepath+'shapefiles/'+site+'-melange-box-UTM'+str(UTMzone)+'.shp'


In [ ]:
# Download the classification masks and reorganize files (https://sentiwiki.copernicus.eu/web/s2-processing#S2Processing-SceneClassification(SC)S2-Processing-Scene-Classificationtrue)
# NO_DATA = 0, CLOUD_MEDIUM_PROBABILITY = 8, CLOUD_HIGH_PROBABILITY = 9
band = 'SCL'

# loop through and download
for tile in tilename:
    tilefolder = tile[0:2]+'/'+tile[2:3]+'/'+tile[3:]+'/'
    for year in years:
        for month in months:
            year = str(year); month = str(int(month)) # convert to strings
            print('Downloading', year, month)
            
            cmd = 'export PATH=/usr/local/bin:/usr/bin:/bin:/usr/sbin:$PATH; '
            cmd += 'aws --no-sign-request s3 cp s3://sentinel-cogs/sentinel-s2-l2a-cogs/'+tilefolder+year+'/'+month+'/'
            cmd += ' '+outputpath+' --recursive --exclude "*/*" --include "*/'+band+'.tif"' # change the band here
            print(cmd)
            subprocess.run(cmd, check=True, shell=True)


# loop through the image subdirectories, rename the SCL.tif file with image identifiers, & move out of subdirectories
for imgfolder in glob.glob(impath+'S2*'):
    if not imgfolder.endswith('.tif') and not imgfolder.endswith('.xml'):
        files = os.listdir(imgfolder)
        for file in files:
            if file == band+'.tif': # rename the files (can change to different band if desired)
                spath = imgfolder+'/'+band+'.tif'
                tpath = impath+imgfolder.split('/')[-1]+'_'+band+'.tif'
                os.rename(spath, tpath)  
                shutil.rmtree(imgfolder)

# crop SCL tiffs with gdalwarp (command line)
for image in os.listdir(impath): 
    # if it hasn't already been clipped
    if not os.path.exists(impath+image[:-4]+band+'_clipped.tif') and image.endswith(band+'.tif'):
        if image.split('_')[2].startswith(('202','201')):
            crop_cmd = 'gdalwarp -cutline '+boxpath+' -crop_to_cutline '+impath+image+" "+impath+image[:-4]+'_clipped.tif'
            print(crop_cmd)
            os.system(crop_cmd)

# DELETE THE ORIGINAL FILES AFTER CLIPPING BECAUSE THEY ARE HUGE! (either here or in terminal)
for image in os.listdir(impath): 
    if not image.endswith(('_clipped.tif','_clipped_10m.tif')):
        os.remove(impath+image)

export PATH=/usr/local/bin:/usr/bin:/bin:/usr/sbin:$PATH; aws --no-sign-request s3 cp s3://sentinel-cogs/sentinel-s2-l2a-cogs/22/W/DE/2019/1/ /Users/ellynenderlin/Research/NSF_GrIS-Freshwater/melange/UMS/images/S2/ --recursive --exclude "*/*" --include "*/SCL.tif"
export PATH=/usr/local/bin:/usr/bin:/bin:/usr/sbin:$PATH; aws --no-sign-request s3 cp s3://sentinel-cogs/sentinel-s2-l2a-cogs/22/W/DE/2019/2/ /Users/ellynenderlin/Research/NSF_GrIS-Freshwater/melange/UMS/images/S2/ --recursive --exclude "*/*" --include "*/SCL.tif"
download: s3://sentinel-cogs/sentinel-s2-l2a-cogs/22/W/DE/2019/2/S2A_22WDE_20190226_0_L2A/SCL.tif to ../../NSF_GrIS-Freshwater/melange/UMS/images/S2/S2A_22WDE_20190226_0_L2A/SCL.tif
download: s3://sentinel-cogs/sentinel-s2-l2a-cogs/22/W/DE/2019/2/S2A_22WDE_20190225_1_L2A/SCL.tif to ../../NSF_GrIS-Freshwater/melange/UMS/images/S2/S2A_22WDE_20190225_1_L2A/SCL.tif
download: s3://sentinel-cogs/sentinel-s2-l2a-cogs/22/W/DE/2019/2/S2B_22WDE_20190226_1_L2A/SCL.tif to ../..

# 2) Identify files that are sufficiently cloud-free that they are worth downloading

In [ ]:
######################################################################################
# Specify the band to download (Sentinel-2 B 8 is NIR)
band = 'B08'

# Adjust cloud thresholds here:
SCLPIXEL_thresh_lower = 8 # medium probability of clouds in Sentinel-2 L2A products
SCLPIXEL_thresh_upper = 9 # high probability of clouds in Sentinel-2 L2A products

cpercent_thresh = 30.0 # maximum cloud cover % in terminus box (default for most downloads is 50)
fpercent_thresh = 60.0 # maximum NO_DATA % in terminus box (default for most downloads is 60)
######################################################################################

In [ ]:
#follow the same methodology as used to identify good Landsat images but using the SCL file, not QA band

#TEMPORARY PATH ADDITION! ONLY NEEDED IF YOU HAVE A TEMPORARY DIRECTORY HOLDING SCL FILES THAT DON'T HAVE DOWNLOADED IMAGES
# impath = '/Volumes/Jokulhaup_5T/Greenland-melange/ASG/images/temp/'
# outputpath = impath

#UTM zones <10 do not lead with a 0 so the bucket path relies on different indexing of the file names than for 2-digit UTM zones
if UTMzone > 9:
    path_idx = 4
else:
    path_idx = 3

#loop through the SCL files
for image in os.listdir(impath): 
    if image.split('_')[-2].startswith('SCL'):
        print(image[:-16])
        SCLpixel = mpimg.imread(impath+image) # read in SCL file as numpy array
        totalpixels = SCLpixel.shape[0]*SCLpixel.shape[1] # count total number of pixels
    
        cloudSCLpixel = SCLpixel[((SCLpixel >= SCLPIXEL_thresh_lower) & (SCLpixel <= SCLPIXEL_thresh_upper))]
        
        # calculate percentages of cloud and fill pixels
        fillpixel = SCLpixel[SCLpixel == 0] 
        cloudpixels = len(cloudSCLpixel); fillpixels = len(fillpixel) # count the cloudy and fill pixels
        cloudpercent = int(float(cloudpixels)/float(totalpixels)*100) # calculate percent cloudy
        fillpercent = int(float(fillpixels)/float(totalpixels)*100) # calculate percent fill
    
        # evaluate thresholds
        if cloudpercent <= cpercent_thresh and fillpercent <= fpercent_thresh:
            # download the band for that scene into your scene folders:
            band = str(band) # string format
            year = str(image[path_idx+6:path_idx+10]); 
            mo = image[path_idx+10:path_idx+12]
            if int(mo) < 10:
                month = str(image[path_idx+11:path_idx+12]) # convert to strings
            else:
                month = str(image[path_idx+10:path_idx+12]) # convert to strings
            tilefolder = image[path_idx:path_idx+2]+'/'+image[path_idx+2]+'/'+image[path_idx+3:path_idx+5]+'/'
            # print(year)
            # print(month)
            # print(tilefolder)
            cmd = 'export PATH=/usr/local/bin:/usr/bin:/bin:/usr/sbin:$PATH; '
            cmd += 'aws --no-sign-request s3 cp s3://sentinel-cogs/sentinel-s2-l2a-cogs/'+tilefolder+year+'/'+month+'/'
            cmd += ' '+outputpath+' --recursive --exclude "*/*" --include "'+image[:-16]+'/'+band+'.tif"' # change the band here
            print(cmd)

            #run the download command
            subprocess.run(cmd, check=True, shell=True)

            # ADDED CROPPING HERE & remove from code below and revise renaming step to look for band+'_clipped.tif'
            crop_cmd = 'gdalwarp -cutline '+boxpath+' -crop_to_cutline '+outputpath+image[:-16]+'/'+band+'.tif'+" "+outputpath+image[:-16]+'/'+band+'_clipped.tif'
            print(crop_cmd)
            os.system(crop_cmd)

            #remove the uncropped image
            os.remove(outputpath+image[:-16]+'/'+band+'.tif')
        else:
            print('failed cloud & fill thresholds: Cloud % ', cloudpercent, 'Fill %', fillpercent)
            # print(impath+image)
            os.remove(impath+image)


In [ ]:
#delete the clipped SCL files
for image in os.listdir(impath):
    if image.split('_')[-2].startswith('SCL'):
        # print(impath+image)
        os.remove(impath+image)

# 4) Sort images:

In [ ]:
# set paths
# imagepath = '/Volumes/JUKES_EXT/CautoRIFT_sites/S2_KG/' # path to the downloaded images
# imagepath = '/Users/jukesliu/Documents/TURNER/DATA/IMAGERY/sentinel2/' # path to the downloaded images
imagepath = impath # path to the downloaded images


In [ ]:
# move out of subdirectories and rename with identifying information
for imgfolder in glob.glob(impath+'S2*'):
    if not imgfolder.endswith('.tif') and not imgfolder.endswith('.xml'):
        files = os.listdir(imgfolder)
        for file in files:
            # if file == band+'.tif': # rename the files (can change to different band if desired)
            #     spath = imgfolder+'/'+band+'.tif'
            #     tpath = impath+imgfolder.split('/')[-1]+'_'+band+'.tif'
            #     os.rename(spath, tpath)  
            #     shutil.rmtree(imgfolder)
            if file == band+'_clipped.tif': # rename the files (can change to different band if desired)
                spath = imgfolder+'/'+band+'_clipped.tif'
                tpath = impath+imgfolder.split('/')[-1]+'_'+band+'_clipped.tif'
                os.rename(spath, tpath)  
                shutil.rmtree(imgfolder)

In [ ]:
# # REPROJECT IF NEEDED: specify the correct UTM zone!!! (326XX for N hemi)
# for image in os.listdir(imagepath): 
#     if not os.path.exists(imagepath+'reprojected/'+image): # if it hasn't already been reprojected
#         rp_cmd = 'gdalwarp -t_srs EPSG:32608 '+imagepath+image+' '+imagepath+'reprojected/'+image
#         os.system(rp_cmd)

# 5) Crop images to AOI: OBSOLETE!!!

In [ ]:
# # crop with gdalwarp (command line) - REPROJECTED IMAGES
# for image in os.listdir(imagepath+'reprojected/'): 
#     # if it hasn't already been clipped
#     if not os.path.exists(imagepath+'reprojected/'+image[:-4]+'_clipped.tif') and image.endswith('.tif'):
#         crop_cmd = 'gdalwarp -cutline '+boxpath+' -crop_to_cutline '+imagepath+'reprojected/'+image+" "+imagepath+'reprojected/'+image[:-4]+'_clipped.tif'
#         print(crop_cmd)
#         os.system(crop_cmd)

...10...20...30...40...50...60...70...80...90...100 - done.
gdalwarp -cutline /Users/jukesliu/Documents/PLANETSCOPE_VELOCITIES/LO/LO_Box_UTM08.shp -crop_to_cutline /Volumes/SURGE_DISK/S2_LO/S2A_8VLM_20221130__B08.tif /Volumes/SURGE_DISK/S2_LO/S2A_8VLM_20221130__B08_clipped.tif
Creating output file that is 4646P x 3921L.
Processing /Volumes/SURGE_DISK/S2_LO/S2A_8VLM_20221130__B08.tif [1/1] : 0Using internal nodata values (e.g. 0) for image /Volumes/SURGE_DISK/S2_LO/S2A_8VLM_20221130__B08.tif.
Copying nodata values from source /Volumes/SURGE_DISK/S2_LO/S2A_8VLM_20221130__B08.tif to destination /Volumes/SURGE_DISK/S2_LO/S2A_8VLM_20221130__B08_clipped.tif.
...10...20...30...40...50...60...70...80...90...100 - done.
gdalwarp -cutline /Users/jukesliu/Documents/PLANETSCOPE_VELOCITIES/LO/LO_Box_UTM08.shp -crop_to_cutline /Volumes/SURGE_DISK/S2_LO/S2B_8VLM_20221102__B08.tif /Volumes/SURGE_DISK/S2_LO/S2B_8VLM_20221102__B08_clipped.tif
Creating output file that is 4646P x 3921L.
Processing /Volum

Creating output file that is 4646P x 3921L.
Processing /Volumes/SURGE_DISK/S2_LO/S2A_8VLM_20220114__B08.tif [1/1] : 0Using internal nodata values (e.g. 0) for image /Volumes/SURGE_DISK/S2_LO/S2A_8VLM_20220114__B08.tif.
Copying nodata values from source /Volumes/SURGE_DISK/S2_LO/S2A_8VLM_20220114__B08.tif to destination /Volumes/SURGE_DISK/S2_LO/S2A_8VLM_20220114__B08_clipped.tif.
...10...20...30...40...50...60...70...80...90...100 - done.


In [ ]:
# OBSOLETE!!! Cropping now done immediately after download
# 
# # crop with gdalwarp (command line)
# for image in os.listdir(imagepath): 
#     # if it hasn't already been clipped
#     if not os.path.exists(imagepath+image[:-4]+band+'_clipped.tif') and image.endswith(band+'.tif'):
#         if image.split('_')[2].startswith(('202','201')):
#             crop_cmd = 'gdalwarp -cutline '+boxpath+' -crop_to_cutline '+imagepath+image+" "+imagepath+image[:-4]+'_clipped.tif'
#             print(crop_cmd)
#             os.system(crop_cmd)

# for image in os.listdir(imagepath): 
#     if not image.endswith(('_clipped.tif','_clipped_10m.tif')):
#         # print(imagepath+image)
#         os.remove(imagepath+image)